In [2]:
import os
import sqlalchemy
from dotenv import load_dotenv
from sqlalchemy import create_engine

load_dotenv()  
user = os.getenv("DB_USER")
password = os.getenv("DB_PASSWORD")
host = os.getenv("DB_HOST")
port = os.getenv("DB_PORT")
db = os.getenv("DB_NAME")

engine = create_engine(f'postgresql+psycopg2://{user}:{password}@{host}:{port}/{db}')

import pandas as pd 
df = pd.read_sql("SELECT * FROM Workers_data;", engine)
df

,age,workclass,fnlgwt,education,educational_num,marital-status,occupation,relationship,race,gender,capital-gain,capital-loss,hours-per-week,native-country,income_>50k
0,67,Private,366425,Doctorate,16,Divorced,Exec-managerial,Not-in-family,White,Male,99999,0,60,United-States,1
1,17,Private,244602,12th,8,Never-married,Other-service,Own-child,White,Male,0,0,15,United-States,0
2,31,Private,174201,Bachelors,13,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,40,United-States,1
3,58,State-gov,110199,7th-8th,4,Married-civ-spouse,Transport-moving,Husband,White,Male,0,0,40,United-States,0
4,25,State-gov,149248,Some-college,10,Never-married,Other-service,Not-in-family,Black,Male,0,0,40,United-States,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
43952,52,Private,68982,Bachelors,13,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,50,United-States,1
43953,19,Private,116562,HS-grad,9,Never-married,Other-service,Own-child,White,Female,0,0,40,United-States,0
43954,30,Private,197947,Some-college,10,Divorced,Sales,Not-in-family,White,Male,0,0,58,United-States,0
43955,46,Private,97883,Bachelors,13,Never-married,Sales,Not-in-family,White,Female,0,0,35,United-States,0


## 1. **DESCRIPTIVE QUERIES**

In [3]:
# How many workers are in the dataset?
df.shape[0]

43957

In [4]:
# What is the gender distribution of workers?
df.gender.value_counts()

gender
Male      29400
Female    14557
Name: count, dtype: int64

In [5]:
# What is the racial distribution of workers?
df.race.value_counts()

race
White                 37572
Black                  4218
Asian-Pac-Islander     1373
Amer-Indian-Eskimo      421
Other                   373
Name: count, dtype: int64

In [8]:
# What are the different education levels and how many workers fall under each?
df.education.value_counts()

education
HS-grad         14197
Some-college     9790
Bachelors        7219
Masters          2392
Assoc-voc        1831
11th             1647
Assoc-acdm       1447
10th             1250
7th-8th           862
Prof-school       748
9th               684
12th              587
Doctorate         536
5th-6th           468
1st-4th           223
Preschool          76
Name: count, dtype: int64

In [7]:
# What is the most common occupation?
df.occupation.value_counts()

occupation
Craft-repair         5519
Prof-specialty       5518
Exec-managerial      5506
Adm-clerical         5004
Sales                4965
Other-service        4448
Machine-op-inspct    2711
Not-Stated           2506
Transport-moving     2121
Handlers-cleaners    1878
Farming-fishing      1348
Tech-support         1321
Protective-serv       874
Priv-house-serv       225
Armed-Forces           13
Name: count, dtype: int64

In [9]:
# What is the most common workclass?
df.workclass.value_counts()

workclass
Private             30587
Self-emp-not-inc     3464
Local-gov            2822
Not-Stated           2498
State-gov            1756
Self-emp-inc         1518
Federal-gov          1284
Without-pay            20
Never-worked            8
Name: count, dtype: int64

In [10]:
# What is the average age of workers?
df.agg(
Average_Age=(
'age','mean')).round(
0).reset_index()

,index,age
0,Average_Age,39.0


In [11]:
# What is the average number of hours worked per week?
df.agg(
Average_Hours_Per_Week=(
'hours-per-week','mean')).round(0).reset_index()

,index,hours-per-week
0,Average_Hours_Per_Week,40.0


In [12]:
# What percentage of workers earn more than $50K?
df.agg(
percentage_Above_50k=(
'income_>50k', lambda x: (
x.sum()/ x.count())*100)).round(
2).reset_index()
# P = ∑x/n​ *100 This is the formula for calculating the percentage of workers earning more than $50k, 
# where ∑x is the sum of workers earning more than $50k, n is the total number of workers, 
# and the result is multiplied by 100 to express it as a percentage.;

,index,income_>50k
0,percentage_Above_50k,23.93


In [13]:
# Which native countries are most represented?
df['native-country'].sort_values().value_counts()

native-country
United-States                 39429
Mexico                          880
Not-Stated                      763
Philippines                     273
Germany                         188
Puerto-Rico                     167
Canada                          158
El-Salvador                     145
India                           134
Cuba                            124
China                           113
England                         109
South                           105
Jamaica                          97
Dominican-Republic               97
Italy                            94
Japan                            83
Guatemala                        79
Vietnam                          77
Columbia                         75
Poland                           72
Haiti                            71
Portugal                         59
Taiwan                           58
Iran                             52
Nicaragua                        46
Greece                           44
Ecuador      

## **2. DIAGONOSTIC / COMPARATIVE QUERIES**

In [14]:
# What percentage of male workers earn more than $50K compared to female workers?
df.groupby(
'gender').agg(
percentage_above_50k=(
'income_>50k', lambda x: (
x.sum()/x.count()) * 100 )).reset_index()

,gender,percentage_above_50k
0,Female,10.922580
1,Male,30.367347


In [15]:
# Which education level has the highest proportion of workers earning above $50K?
df.groupby(
'education').agg(
earning_above_50k=('income_>50k', 'sum')).sort_values(
by='earning_above_50k', ascending= False).reset_index()

,education,earning_above_50k
0,Bachelors,2989
1,HS-grad,2224
2,Some-college,1850
3,Masters,1328
4,Prof-school,551
5,Assoc-voc,469
6,Assoc-acdm,385
7,Doctorate,384
8,11th,84
9,10th,82


In [17]:
# Which occupation has the most workers earning above $50K?
df.groupby(
'occupation').agg(
Earnings_Above_50k=(
'income_>50k','sum')).sort_values(
by= 'Earnings_Above_50k', ascending=False).reset_index()

,occupation,Earnings_Above_50k
0,Exec-managerial,2633
1,Prof-specialty,2496
2,Sales,1337
3,Craft-repair,1259
4,Adm-clerical,668
5,Transport-moving,420
6,Tech-support,383
7,Machine-op-inspct,328
8,Protective-serv,280
9,Not-Stated,236


In [20]:
# How does average hours per week differ across occupations?
df.groupby(
'occupation').agg(
Average_Hours_Per_Week=(
'hours-per-week','mean')).round(2).sort_values(
by= 'Average_Hours_Per_Week', ascending= False
).reset_index()

,occupation,Average_Hours_Per_Week
0,Farming-fishing,46.49
1,Exec-managerial,44.98
2,Transport-moving,44.82
3,Armed-Forces,44.31
4,Protective-serv,42.67
5,Prof-specialty,42.27
6,Craft-repair,42.25
7,Sales,40.80
8,Machine-op-inspct,40.77
9,Tech-support,39.56


In [21]:
# Which workclass has the highest proportion of high earners?
df.groupby(
'workclass').agg(
Earners=(
'income_>50k',lambda x: (
x.sum()/x.count())
* 100)).sort_values(
by ='Earners', ascending = False
).round(0).reset_index()

,workclass,Earners
0,Self-emp-inc,56.0
1,Federal-gov,39.0
2,Local-gov,30.0
3,Self-emp-not-inc,28.0
4,State-gov,27.0
5,Private,22.0
6,Without-pay,10.0
7,Not-Stated,9.0
8,Never-worked,0.0


In [22]:
# How does marital status relate to income level?
df.groupby(
'marital-status').agg(
Percentage_Per_Income = (
'income_>50k', lambda x: (
x.sum()/x.count())
* 100
)
).sort_values(
by = 'Percentage_Per_Income', ascending =  False).round(0).reset_index()

,marital-status,Percentage_Per_Income
0,Married-civ-spouse,45.0
1,Married-AF-spouse,35.0
2,Divorced,10.0
3,Married-spouse-absent,9.0
4,Widowed,8.0
5,Separated,6.0
6,Never-married,5.0


In [27]:
# What is the average age of workers who earn above $50K vs those who do not?
df.groupby(
'income_>50k'
).agg(
Average_Age = (
'age', 'mean'
)
).round(0).reset_index()

,income_>50k,Average_Age
0,0,37.0
1,1,44.0


In [24]:
# Which race group has the highest proportion of workers earning above $50K?
df.groupby(
'race'
).agg(
Proportion_Of_Workers = (
'income_>50k', lambda x:
(
x.sum()/x.count()
) * 100
)
).sort_values(
by = 'Proportion_Of_Workers', ascending = False
).round(0).reset_index()

,race,Proportion_Of_Workers
0,Asian-Pac-Islander,27.0
1,White,25.0
2,Black,12.0
3,Other,12.0
4,Amer-Indian-Eskimo,11.0


In [25]:
# Do workers with more education work more hours per week?
df.groupby(
'education').agg(
Working_Hours_Per_Week = (
'hours-per-week','sum'
)
).sort_values(
by = 'Working_Hours_Per_Week', ascending = False
).reset_index()

,education,Working_Hours_Per_Week
0,HS-grad,577521
1,Some-college,380106
2,Bachelors,306555
3,Masters,104258
4,Assoc-voc,76171
5,Assoc-acdm,59026
6,11th,55665
7,10th,45940
8,Prof-school,35910
9,7th-8th,33379


## **3. FILTERING / SLICING QUERIES**


In [26]:
# Which native country has the highest proportion of high earners outside the US?
Non_US_workers = df[df['native-country'] != 'United-States']
Non_US_workers.groupby('native-country').agg(
    Earners=('income_>50k', lambda x: (x.sum()/x.count())*100),
    Count=('income_>50k', 'count')
).sort_values(by='Earners', ascending=False).round(0).reset_index().head(5)

,native-country,Earners,Count
0,India,42.0,134
1,France,41.0,32
2,Taiwan,38.0,58
3,Greece,36.0,44
4,England,36.0,109


In [28]:
# How many workers work more than 60 hours per week?
work=df[df['hours-per-week']>60]
work.groupby(
'workclass').agg(
work_hours_per_week = (
'hours-per-week','count', 
)
).sort_values(
by = 'work_hours_per_week', ascending = False
).reset_index()

,workclass,work_hours_per_week
0,Private,770
1,Self-emp-not-inc,349
2,Self-emp-inc,170
3,Local-gov,81
4,State-gov,54
5,Not-Stated,51
6,Federal-gov,26
7,Without-pay,1


In [29]:
# How many workers have a Doctorate or Masters degree?
degree = df[df['education'].isin(['Doctorate', 'Masters'])]
degree.groupby(
'education'
).agg(
Number_of_workers=(
'education','count'
)
).sort_values(
by='Number_of_workers', ascending=False
).reset_index()

,education,Number_of_workers
0,Masters,2392
1,Doctorate,536


In [31]:
# Who are the workers above age 60 still working in the Private sector?
aged_workers = df[(df['age'] > 60) & (df['workclass'] == 'Private')]
aged_workers.groupby(
	'workclass'
).agg(
	Workers_above_60=('workclass', 'count')
).reset_index()


,workclass,Workers_above_60
0,Private,1521


In [32]:
print(aged_workers)

       age workclass  fnlgwt     education  educational_num  \
0       67   Private  366425     Doctorate               16   
6       70   Private  216390           9th                5   
15      76   Private  316185       7th-8th                4   
32      71   Private  152307       HS-grad                9   
40      62   Private  312818     Bachelors               13   
...    ...       ...     ...           ...              ...   
43845   61   Private  176839       HS-grad                9   
43849   65   Private  195568  Some-college               10   
43875   62   Private  319582       HS-grad                9   
43882   79   Private  149912     Bachelors               13   
43928   65   Private  461715       HS-grad                9   

           marital-status         occupation   relationship   race  gender  \
0                Divorced    Exec-managerial  Not-in-family  White    Male   
6      Married-civ-spouse  Machine-op-inspct           Wife  White  Female   
15       

In [33]:
# How many female workers earn above $50K?
Female_Workers = df[df['gender']== 'Female']
Female_Workers.groupby(
'gender'
).agg(
Earners = (
'income_>50k', 'sum')
).sort_values(
by = 'Earners', ascending = False
).reset_index()

,gender,Earners
0,Female,1590


In [34]:
# Which workers have capital gain greater than zero?
Capital_gain = df[df['capital-gain'] > 0]
Capital_gain.groupby(
'workclass').agg(
Capital_Gain = (
'capital-gain','count'
)
).sort_values(
by = 'Capital_Gain', ascending = False
).reset_index()

,workclass,Capital_Gain
0,Private,2322
1,Self-emp-not-inc,338
2,Self-emp-inc,282
3,Local-gov,254
4,Not-Stated,155
5,State-gov,138
6,Federal-gov,136
7,Without-pay,2
